# Task
Mount Google Drive and create a 4x3 Plotly subplot grid showing population and urbanisation share over time for each region in "./csv/regional_urbanisation.csv". Display rural_share and urban_share as percentages on the right y-axis and rural_population, urban_population, and total_population on the left y-axis.

## Mount google drive

### Subtask:
Mount Google Drive to access the file.


**Reasoning**:
Mount Google Drive to access the file.



In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import plotly.io as pio
pio.renderers.default = "notebook"  # or "notebook_connected", "iframe", etc.

import plotly.graph_objs as go

fig = go.Figure(go.Scatter(x=[1,2,3], y=[4,5,6]))
fig.show()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import plotly.graph_objs as go
from plotly.subplots import make_subplots

# Find the CSV file using os.walk
csv_path = None
for root, dirs, files in os.walk('/content/drive'):
    for file in files:
        if file == 'regional_urbanisation.csv':
            csv_path = os.path.join(root, file)
            break
    if csv_path:
        break

if not csv_path:
    raise FileNotFoundError("regional_urbanisation.csv not found in Google Drive.")

# Load the CSV
df = pd.read_csv(csv_path)

# Ensure correct types
df['year'] = df['year'].astype(int)
df['rural_share'] = df['rural_share'].astype(float)
df['urban_share'] = df['urban_share'].astype(float)
df['rural_population'] = df['rural_population'].astype(float)
df['urban_population'] = df['urban_population'].astype(float)
df['total_population'] = df['total_population'].astype(float)

# Get unique regions
regions = sorted(df['region'].unique())
n_regions = len(regions)
n_rows = 4
n_cols = 3

# Get global year range for equal interval x-axis
min_year = df['year'].min()
max_year = df['year'].max()
year_tick = 5 if (max_year - min_year) > 20 else 1

# Color schemes (solid lines)
pop_colors = ['#1f77b4', '#2ca02c', '#ff7f0e']  # blue, green, orange
share_colors = ['#d62728', '#9467bd']           # red, purple

# Create subplots
fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=regions,
    shared_xaxes=False,
    shared_yaxes=False,
    vertical_spacing=0.08,
    horizontal_spacing=0.05,
    specs=[[{"secondary_y": True} for _ in range(n_cols)] for _ in range(n_rows)]
)

# Plot for each region
for idx, region in enumerate(regions):
    region_df = df[df['region'] == region].sort_values('year')
    row = idx // n_cols + 1
    col = idx % n_cols + 1

    # Left axis: population counts (solid lines, varied colors)
    fig.add_trace(
        go.Scatter(
            x=region_df['year'],
            y=region_df['rural_population'],
            name='Rural Population' if idx == 0 else None,
            mode='lines+markers',
            line=dict(color=pop_colors[0], width=2),
            marker=dict(size=5),
            showlegend=(idx == 0)
        ),
        row=row,
        col=col,
        secondary_y=False
    )
    fig.add_trace(
        go.Scatter(
            x=region_df['year'],
            y=region_df['urban_population'],
            name='Urban Population' if idx == 0 else None,
            mode='lines+markers',
            line=dict(color=pop_colors[1], width=2),
            marker=dict(size=5),
            showlegend=(idx == 0)
        ),
        row=row,
        col=col,
        secondary_y=False
    )
    fig.add_trace(
        go.Scatter(
            x=region_df['year'],
            y=region_df['total_population'],
            name='Total Population' if idx == 0 else None,
            mode='lines+markers',
            line=dict(color=pop_colors[2], width=2),
            marker=dict(size=5),
            showlegend=(idx == 0)
        ),
        row=row,
        col=col,
        secondary_y=False
    )

    # Right axis: shares as percentages (solid lines, varied colors)
    fig.add_trace(
        go.Scatter(
            x=region_df['year'],
            y=region_df['rural_share'] * 100,
            name='Rural Share (%)' if idx == 0 else None,
            mode='lines+markers',
            line=dict(color=share_colors[0], width=2),
            marker=dict(size=5),
            showlegend=(idx == 0)
        ),
        row=row,
        col=col,
        secondary_y=True
    )
    fig.add_trace(
        go.Scatter(
            x=region_df['year'],
            y=region_df['urban_share'] * 100,
            name='Urban Share (%)' if idx == 0 else None,
            mode='lines+markers',
            line=dict(color=share_colors[1], width=2),
            marker=dict(size=5),
            showlegend=(idx == 0)
        ),
        row=row,
        col=col,
        secondary_y=True
    )

    # Set equal interval x-axis for each subplot
    fig.update_xaxes(
        range=[min_year, max_year],
        dtick=year_tick,
        row=row,
        col=col,
        title_text="Year"
    )

    # Set y-axes titles
    fig.update_yaxes(
        title_text="Population",
        row=row,
        col=col,
        secondary_y=False
    )
    fig.update_yaxes(
        title_text="Share (%)",
        row=row,
        col=col,
        secondary_y=True
    )

fig.update_layout(
    height=1800,
    width=1800,
    title_text="Regional Urbanisation: Population & Share Over Time",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    margin=dict(t=80)
)

fig.show()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
